# Final ML Problem Formulation
- Given a request time and trip context, predict the pickup zone where demand is likely to occur.

| Aspect   | Value                            |
| -------- | -------------------------------- |
| Type     | Supervised Learning              |
| Category | Multi-Class Classification       |
| Target   | `pickup_zone`                    |
| Classes  | High cardinality (100s of zones) |


In [1]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier


In [2]:
df = pd.read_csv(
    r"E:\Personal Things\02 Projects\23 Ride hailing\Data Processing\yellow_tripdata_2015-01.csv"
)

df = df.sample(n=1_00_000, random_state=42)  # use ~15 lakh rows
df.shape


(100000, 19)

In [3]:
# Datetime Conversion
df["pickup_time"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["drop_time"] = pd.to_datetime(df["tpep_dropoff_datetime"])


In [4]:
# Request Time
df["request_time"] = df["pickup_time"] - pd.to_timedelta(
    np.random.randint(2, 8, size=len(df)), unit="m"
)


In [5]:
# Distance & Duration
df["trip_distance_km"] = df["trip_distance"] * 1.60934

df["trip_duration_min"] = (
    (df["drop_time"] - df["pickup_time"]).dt.total_seconds() / 60
)


In [6]:
# Hard Logical Cleaning
df = df[
    (df["trip_distance_km"].between(0.1, 100)) &
    (df["trip_duration_min"].between(1, 300)) &
    (df["fare_amount"].between(2, 500)) &
    (df["total_amount"].between(2, 600))
]


In [7]:
# GPS Boundary Cleaning (NYC)
df = df[
    (df["pickup_latitude"].between(40.5, 41.0)) &
    (df["pickup_longitude"].between(-74.3, -73.6))
]


In [8]:
# Log Transformation (Skewness Fix)
df["trip_distance_log"] = np.log1p(df["trip_distance_km"])
df["trip_duration_log"] = np.log1p(df["trip_duration_min"])
df["fare_amount_log"] = np.log1p(df["fare_amount"])


In [9]:
# Time Features
df["request_hour"] = df["request_time"].dt.hour
df["request_weekday"] = df["request_time"].dt.weekday
df["is_weekend"] = df["request_weekday"].isin([5, 6]).astype(int)


In [10]:
# Payment Type Frequency Encoding
payment_freq = df["payment_type"].value_counts(normalize=True)
df["payment_type_freq"] = df["payment_type"].map(payment_freq)

In [11]:
# Pickup Zone Creation (KMeans, COARSE)
kmeans = KMeans(
    n_clusters=20,
    random_state=42,
    n_init=10
)

df["pickup_zone"] = kmeans.fit_predict(
    df[["pickup_latitude", "pickup_longitude"]]
)

df["pickup_zone"] = "Zone_" + df["pickup_zone"].astype(str)


In [12]:
# Zone Demand Frequency (Strong Signal)
zone_freq = df["pickup_zone"].value_counts(normalize=True)
df["zone_demand_freq"] = df["pickup_zone"].map(zone_freq)


In [21]:
# Final Feature Selection & Cleaning
feature_cols = [
    "request_hour",
    "request_weekday",
    "is_weekend",
    "passenger_count",
    "RateCodeID",
    "payment_type_freq",
    "trip_distance_log",
    "trip_duration_log",
    "fare_amount_log"
]


df_clean = (
    df[feature_cols + ["pickup_zone"]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
)

X = df_clean[feature_cols]
y = df_clean["pickup_zone"]



In [22]:
# Define X and y (FROM SCRATCH)
X = df_clean[feature_cols]
y = df_clean["pickup_zone"]

X.shape, y.shape


((97216, 9), (97216,))

In [23]:
# Train–Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [24]:
# Train LightGBM Model
model = LGBMClassifier(
    objective="multiclass",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=128,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002347 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 746
[LightGBM] [Info] Number of data points in the train set: 77772, number of used features: 9
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[LightGBM] [Info] Start training from score -2.995732
[

LGBMClassifier(class_weight='balanced', colsample_bytree=0.8,
               learning_rate=0.05, n_estimators=400, num_leaves=128,
               objective='multiclass', random_state=42, subsample=0.8)

In [ ]:
# Accuracy
y_pred = model  .predict(X_test)
accuracy_score(y_test, y_pred)


0.13778029212096277

In [26]:
# Top-5 Accuracy (PRIMARY METRIC)
y_proba = model.predict_proba(X_test)

top5 = np.argsort(y_proba, axis=1)[:, -5:]

top5_accuracy = np.mean([
    y_test.iloc[i] in model.classes_[top5[i]]
    for i in range(len(y_test))
])

top5_accuracy


0.502314338613454

In [19]:
# Single Prediction Test
sample = X_test.iloc[[0]]
model.predict(sample)


array(['Zone_14'], dtype=object)

In [20]:
# Top-5 Zone Recommendation
proba = model.predict_proba(sample)[0]
model.classes_[np.argsort(proba)[-5:]]


array(['Zone_0', 'Zone_8', 'Zone_15', 'Zone_18', 'Zone_14'], dtype=object)